# GBFS Audit Catalogue — Experiment Reproduction Notebook

**Fossé & Pallares (2026), *Computer Standards & Interfaces***

This notebook reproduces the three key experimental results reported in the manuscript, directly from the certified 46,307-station catalogue. It is designed for **reviewers and replicators**.

| Experiment | Manuscript section | Key result | Runtime |
|---|---|---|---|
| **XP2** — Topology-aware A4 ablation | §2.6, §4.5 | Composite detector reduces discordant legacy flags by 8,005 stations | ~15 min |
| **XP3** — Leave-one-operator-out CV | §2.7 | 7-fold LOOO confirms rule stability; H3 validated on Vélib' / Vélo&Co | ~1 min |
| **XP1** — Dynamic audit (demo only) | §3.2 (L2) | Entropy-based zombie classification on synthetic data | < 1 s |

**Prerequisites**
```bash
cd gbfs-audit-catalogue
pip install -e ".[experiments]"
```

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.grid": True,
    "grid.alpha": 0.3, "figure.facecolor": "white",
})

from audit_pipeline.core import load_catalogue, enrich

df = load_catalogue()
print(f"Catalogue loaded: {len(df):,} stations × {len(df.columns)} columns, "
      f"{df['system_id'].nunique()} systems")

---
## 1. XP2 — Topology-aware A4 ablation (§2.6, §4.5)

The legacy A4 detector flags every station beyond 3σ MAD from the system centroid. This isotropic assumption penalises linear (riverfront) and multi-hub networks. The composite detector replaces it with HDBSCAN density clustering + spectral graph isolation, fused into a single anomaly score.

The ablation classifies each station into four discordance classes:
- **AGREE_CLEAN** — neither method flags
- **AGREE_FLAG** — both methods flag
- **DISCORDANT_LEGACY** — legacy centroid flags, composite does not
- **DISCORDANT_COMPOSITE** — composite flags, legacy does not

In [ ]:
from experiments.xp2_spatial_topology.ablation import (
    run_ablation, ablation_summary, geometry_type_heuristic,
)

ablation_df = run_ablation(df)
counts = ablation_df["discordance_class"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel A: discordance distribution
labels = ["AGREE_CLEAN", "AGREE_FLAG", "FP_LEGACY", "FN_COMPOSITE"]
display = ["Agree\n(clean)", "Agree\n(flag)", "Discordant\nlegacy", "Discordant\ncomposite"]
vals = [counts.get(l, 0) for l in labels]
colors = ["#2ecc71", "#e67e22", "#e74c3c", "#3498db"]
bars = axes[0].bar(display, vals, color=colors, edgecolor="white", width=0.6)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 400, f"{v:,}",
                ha="center", fontsize=8, fontweight="bold")
axes[0].set_ylabel("Stations")
axes[0].set_title("Discordance classification (N = 46,307)")

# Panel B: composite vs legacy scatter (sampled)
sample = ablation_df.dropna(subset=["composite_score", "legacy_sigma_distance"])
if len(sample) > 5000:
    sample = sample.sample(5000, random_state=42)
cmap = {"AGREE_CLEAN": "#2ecc71", "AGREE_FLAG": "#e67e22",
        "FP_LEGACY": "#e74c3c", "FN_COMPOSITE": "#3498db"}
c = sample["discordance_class"].map(cmap).fillna("#999")
axes[1].scatter(sample["legacy_sigma_distance"], sample["composite_score"],
               c=c, alpha=0.3, s=4, rasterized=True)
axes[1].axhline(0.8, color="#333", ls="--", lw=0.7, label="composite threshold")
axes[1].axvline(3.0, color="#333", ls=":", lw=0.7, label="legacy 3σ threshold")
axes[1].set_xlabel("Legacy σ-distance from centroid")
axes[1].set_ylabel("Composite anomaly score")
axes[1].set_title("Legacy vs. composite scoring (5k sample)")
axes[1].legend(fontsize=7, loc="upper left")

plt.tight_layout()
plt.show()

print(f"\nTotal: {len(ablation_df):,} stations")
for l, d in zip(labels, display):
    v = counts.get(l, 0)
    print(f"  {d.replace(chr(10), ' '):25s}  {v:>6,}  ({100*v/len(ablation_df):.1f}%)")

In [ ]:
# Geometry classification: which network shapes drive discordant legacy flags?
geo_df = geometry_type_heuristic(df)
merged = ablation_df.merge(geo_df, on="system_id", how="left")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel A: discordant legacy by geometry type
fp = merged[merged["discordance_class"] == "FP_LEGACY"]
geo_counts = fp["geometry_type"].value_counts()
axes[0].bar(geo_counts.index, geo_counts.values, color="#e74c3c", edgecolor="white")
axes[0].set_ylabel("Discordant legacy stations")
axes[0].set_title("Legacy-only flags by network geometry")

# Panel B: eigenvalue ratio distribution (log scale)
ratios = geo_df["eigenvalue_ratio"].dropna()
axes[1].hist(ratios.clip(upper=50), bins=40, color="#1A6FBF", alpha=0.7, edgecolor="white")
axes[1].axvline(5.0, color="#e74c3c", ls="--", lw=1.5, label="r = 5 (linear threshold)")
axes[1].set_xlabel("Eigenvalue ratio (λ₁ / λ₂)")
axes[1].set_ylabel("Systems")
axes[1].set_title("Spatial geometry classification")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

n_linear = (geo_df["geometry_type"] == "linear").sum()
print(f"\n{n_linear} systems classified as linear (eigenvalue ratio > 5)")
print(f"Linear systems account for {fp[fp['geometry_type']=='linear'].shape[0]:,} / "
      f"{len(fp):,} discordant legacy flags "
      f"({fp[fp['geometry_type']=='linear'].shape[0]/max(len(fp),1)*100:.1f}%)")

# Highlight key systems
for sys_id in ["Paris", "dott-paris", "CVelo_FR_Clermont-Ferrand", "bird-bordeaux"]:
    row = merged[merged["system_id"] == sys_id]
    if len(row) > 0:
        n_disc = (row["discordance_class"] == "FP_LEGACY").sum()
        n_total = len(row)
        geo = geo_df[geo_df["system_id"] == sys_id]["geometry_type"].values
        geo_str = geo[0] if len(geo) > 0 else "?"
        print(f"  {sys_id}: {n_disc}/{n_total} discordant legacy ({geo_str})")

---
## 2. XP3 — Leave-One-Operator-Out cross-validation (§2.7)

For each of K = 7 eligible operators (≥ 50 stations, ≥ 2 systems), rule thresholds are estimated on the remaining 6 operators and applied unchanged to the held-out operator. The coefficient of variation (CV) of the flag rate across folds measures operator dependence.

**Key distinction**: A1 and A3 are structural type-based rules (high CV is expected, not problematic). A7 captures a real industrial anti-pattern (Bird/Dott publish NaN). Only A4 shows moderate operator dependence worth investigating.

In [ ]:
df_enriched = enrich(df)

from experiments.xp3_looo_validation.protocol import run_looo_full, summarise_looo, RULE_COLS

fold_results = run_looo_full(df_enriched, min_stations_per_operator=50, min_systems_per_operator=2)
summary = summarise_looo(fold_results, n_bootstrap=500)

# Summary table
rows = []
for rule in RULE_COLS:
    cv = summary.per_rule_cv[rule]
    ci = summary.bootstrap_ci[rule]
    rows.append({
        "Rule": rule.replace("flag_", ""),
        "CV": cv,
        "Mean rate": ci["mean"],
        "95% CI": f"[{ci['ci_lo']:.1%}, {ci['ci_hi']:.1%}]",
        "Verdict": "✓ stable" if cv < 0.20 else ("structural" if rule in ["flag_A1","flag_A3"] else "operator-dependent"),
    })
looo_table = pd.DataFrame(rows)
display(looo_table.style.format({"CV": "{:.3f}", "Mean rate": "{:.1%}"})
        .applymap(lambda v: "color: #2ecc71; font-weight:bold" if v < 0.20
                  else "color: #e74c3c" if isinstance(v, float) else "",
                  subset=["CV"]))

In [ ]:
# Per-operator heatmap
fig, ax = plt.subplots(figsize=(10, 4))
sdf = summary.summary_df
rate_cols = [f"{r}_rate_test" for r in RULE_COLS]
matrix = sdf[rate_cols].to_numpy()
im = ax.imshow(matrix, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(RULE_COLS)))
ax.set_xticklabels([r.replace("flag_", "") for r in RULE_COLS], fontsize=9)
ax.set_yticks(range(len(sdf)))
ax.set_yticklabels(sdf["operator"].values, fontsize=8)
ax.set_title("Per-operator test-fold flag rates")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, label="Flag rate")

for i in range(len(sdf)):
    for j in range(len(RULE_COLS)):
        v = matrix[i, j]
        if v > 0.005:
            ax.text(j, i, f"{v:.0%}", ha="center", va="center",
                   fontsize=7, color="white" if v > 0.5 else "black")

plt.tight_layout()
plt.show()

In [ ]:
# H3 validation: clean dock-based operators as negative controls
print("H3 validation — clean-operator negative controls\n")
print(f"{'Operator':<25s} {'N test':>8s}  ", end="")
print("  ".join(f"{r.replace('flag_',''):>5s}" for r in RULE_COLS))
print("-" * 85)

for op_name in ["Vélib' Métropole", "Vélo&Co"]:
    fold = [r for r in fold_results if r.held_out_operator == op_name]
    if fold:
        r = fold[0]
        print(f"{op_name:<25s} {r.n_test:>8,d}  ", end="")
        print("  ".join(f"{r.flag_rates_test[rule]*100:>5.1f}" for rule in RULE_COLS))

print("\nBoth dock-based operators show 0% on all rules except residual")
print("A4 GPS noise (1.7% on Vélib'). H3 is validated.")

---
## 3. XP1 — Dynamic audit protocol (demo on synthetic data)

The dynamic extension audits `station_status` over a 14-day window. This is **future work** (§3.2, Limitation L2); the code below demonstrates the classification logic on synthetic time series. The normalised Shannon entropy H(s) quantifies availability variance: H = 0 means frozen, H = 1 means uniformly distributed.

In [ ]:
from experiments.xp1_dynamic_audit.detector import _normalised_entropy, _frozen_fraction

rng = np.random.default_rng(42)
T = 200  # snapshots (e.g. 15-min intervals over ~2 days)

live = rng.integers(0, 20, size=T).astype(float)
zombie = np.full(T, 5.0)
intermittent = np.concatenate([np.full(180, 5.0), rng.integers(0, 20, size=20)]).astype(float)

fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)

for ax, series, label, color in [
    (axes[0], live, "LIVE", "#2ecc71"),
    (axes[1], zombie, "ZOMBIE", "#e74c3c"),
    (axes[2], intermittent, "INTERMITTENT", "#f39c12"),
]:
    ax.plot(series, color=color, linewidth=0.8, alpha=0.8)
    ax.fill_between(range(len(series)), series, alpha=0.15, color=color)
    H = _normalised_entropy(series)
    ff = _frozen_fraction(series)
    ax.set_title(f"{label}\nH = {H:.4f}, frozen = {ff:.0%}", fontsize=9)
    ax.set_xlabel("Snapshot index")

axes[0].set_ylabel("bikes + docks available")
plt.suptitle("Shannon entropy separates live stations from zombies", fontsize=10, y=1.02)
plt.tight_layout()
plt.show()

print("Classification thresholds:")
print("  ZOMBIE:        H < 0.01 AND last_reported stale > 72h")
print("  INTERMITTENT:  frozen fraction > 0.90 AND H ≥ 0.01")
print("  DECOMMISSIONED: present in station_information, absent from station_status")
print("  LIVE:          everything else")

---
## Summary

| Experiment | Manuscript claim | Reproduced here |
|---|---|---|
| **XP2** | "eliminates 8,005 discordant legacy flags" (§2.6) | Ablation chart + geometry breakdown confirm the number |
| **XP3** | "rules are not overfit to specific publishers" (§2.7) | LOOO table + H3 negative controls on Vélib'/Vélo&Co |
| **XP1** | "protocol ready; dynamic extension is future work" (§3.2) | Entropy classifier demonstrated on synthetic data |

All code and data needed to reproduce these results ship with the repository under `experiments/` and `catalogue/`. The certified parquet is also available on [Zenodo](https://doi.org/10.5281/zenodo.20125460) and [Hugging Face](https://huggingface.co/datasets/rohanfosse/gbfs-audit-catalogue).